# 03 - Training SRCNN for Underwater Super-Resolution

This notebook demonstrates how to:
1. Instantiate the SRCNN network for 2x or 4x upscaling.
2. Configure the loss function (MSE Loss) and optimizer (Adam).
3. Run training iterations with learning rate scheduling.
4. Save model checkpoints.

In [ ]:
import sys
sys.path.append("../src")

import torch
import torch.nn as nn
import torch.optim as optim
from underwater_sr.model import SRCNN
from underwater_sr.metrics import calculate_psnr
from underwater_sr.utils import get_device

device = get_device("auto")
print(f"Using device: {device}")

### Initialize Model

In [ ]:
scale_factor = 4
model = SRCNN(scale_factor=scale_factor).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")

### Step-by-Step Training Step on Synthetic Batch
Demonstrating forward pass, loss computation, backward pass, and optimizer update.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Create synthetic batch (Batch=4, Channels=3, H=32, W=32 for LR)
dummy_lr = torch.rand(4, 3, 32, 32, device=device)
dummy_hr = torch.rand(4, 3, 128, 128, device=device)

model.train()
optimizer.zero_grad()

pred_sr = model(dummy_lr)
loss = criterion(pred_sr, dummy_hr)
loss.backward()
optimizer.step()

psnr_val = calculate_psnr(pred_sr, dummy_hr)
print(f"Iteration Loss: {loss.item():.5f} | Initial Batch PSNR: {psnr_val:.2f} dB")
print(f"Output dimensions verified: {pred_sr.shape}")